In [2]:
# Cell 1: Google Drive mount (a CPU runtime is enough for Stage 5)
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# Cell 1b: Label-fraction files on Drive: results, predictions and checkpoint folders
from pathlib import Path
import json

CONTENT = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content")

print("Results files (frac10):")
for f in sorted((CONTENT / "results").glob("*frac10*")):
    print("   ", f.name)

print("\nPrediction files (frac10):")
for f in sorted((CONTENT / "predictions").glob("*frac10*")):
    print("   ", f.name)

print("\nCheckpoint folders (frac10): files and last saved epoch")
for d in sorted((CONTENT / "checkpoints").glob("*frac10*")):
    files = sorted(p.name for p in d.iterdir())
    history = d / "history.json"
    last_epoch = json.loads(history.read_text())[-1]["epoch"] if history.exists() else None
    print(f"    {d.name}: {files}, last epoch = {last_epoch}")

Results files (frac10):
    prithvi_frozen_frac10_seed0.json
    subset_frac10.json
    unet_s2_6_frac10_seed0.json
    unet_s2_6_frac10_seed1.json

Prediction files (frac10):
    prithvi_frozen_frac10_seed0_bolivia.npz
    unet_s2_6_frac10_seed0_bolivia.npz
    unet_s2_6_frac10_seed1_bolivia.npz

Checkpoint folders (frac10): files and last saved epoch
    prithvi_frozen_frac10_seed0: ['best.pt', 'history.json', 'last.pt'], last epoch = 100
    prithvi_frozen_frac10_seed1: ['best.pt', 'history.json', 'last.pt'], last epoch = 72
    prithvi_frozen_frac10_seed2: [], last epoch = None
    unet_s2_6_frac10_seed0: ['best.pt', 'history.json', 'last.pt'], last epoch = 100
    unet_s2_6_frac10_seed1: ['best.pt', 'history.json', 'last.pt'], last epoch = 100
    unet_s2_6_frac10_seed2: [], last epoch = None


 Cell 2 reads all 24 results files and puts the TP, FP and FN of each chip into arrays, with the same chip order for all methods.

In [3]:
# Cell 2: Results files, method list and per-chip counts in a fixed chip order
import json
from pathlib import Path
import numpy as np
import pandas as pd

RESULTS_DIR = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/results")
SPLITS = ["test", "bolivia"]

# Method key: (label, results files of its seeds)
METHODS = {
    "otsu": ("Otsu (VH)", ["otsu_vh"]),
    "fixed": ("Fixed VH threshold", ["fixed_vh"]),
    "unet_s1": ("U-Net radar, 2 bands", [f"unet_s1_seed{s}" for s in range(3)]),
    "unet_s2": ("U-Net optical, 13 bands", [f"unet_s2_seed{s}" for s in range(3)]),
    "unet_s2_6": ("U-Net optical, 6 bands", [f"unet_s2_6_seed{s}" for s in range(3)]),
    "prithvi_frozen": ("Prithvi frozen, 6 bands", [f"prithvi_frozen_seed{s}" for s in range(3)]),
    "prithvi_full": ("Prithvi full, 6 bands", ["prithvi_full_seed0"]),
    "unet_s2_6_frac10": ("U-Net optical, 6 bands, 10% labels", [f"unet_s2_6_frac10_seed{s}" for s in range(3)]),
    "prithvi_frozen_frac10": ("Prithvi frozen, 6 bands, 10% labels", [f"prithvi_frozen_frac10_seed{s}" for s in range(3)]),
}

results = {name: json.loads((RESULTS_DIR / f"{name}.json").read_text())
           for _, files in METHODS.values() for name in files}
print("Results files read:", len(results))

# Fixed chip order for each split; every results file must contain exactly these chips
CHIPS = {split: sorted(results["fixed_vh"]["splits"][split]["chips"]) for split in SPLITS}
for name, r in results.items():
    for split in SPLITS:
        assert sorted(r["splits"][split]["chips"]) == CHIPS[split], f"Chip mismatch: {name}, {split}"

def counts_array(name, split):
    """Per-chip TP, FP, FN of one results file, shape (n_chips, 3), in the fixed chip order."""
    chips = results[name]["splits"][split]["chips"]
    return np.array([[chips[c]["tp"], chips[c]["fp"], chips[c]["fn"]] for c in CHIPS[split]], dtype=np.int64)

# Counts of each method and split, shape (n_seeds, n_chips, 3)
COUNTS = {key: {split: np.stack([counts_array(n, split) for n in files]) for split in SPLITS}
          for key, (_, files) in METHODS.items()}

for split in SPLITS:
    print(f"{split}: {len(CHIPS[split])} chips")

Results files read: 21
test: 90 chips
bolivia: 15 chips


 Cell 3 makes 10,000 random chip lists for each split, the same lists for all methods. Then it calculates the seed-mean IoU of each method with a 95% interval, and saves the final table.

In [4]:
# Cell 3: Chip bootstrap (10,000 samples): seed-mean IoU with 95% interval for each method; final table saved
B = 10_000
rng = np.random.default_rng(0)

# The same resampled chip lists are used for every method, so that paired differences are valid
BOOT_IDX = {split: rng.integers(0, len(CHIPS[split]), size=(B, len(CHIPS[split]))) for split in SPLITS}

def seed_ious(counts):
    """IoU of each seed from counts summed over chips. counts: (n_seeds, n_chips, 3)."""
    tp, fp, fn = counts.sum(axis=1).T
    return tp / (tp + fp + fn)

def boot_seed_mean_iou(counts, idx):
    """Seed-mean IoU for each bootstrap sample. counts: (n_seeds, n_chips, 3); idx: (B, n_chips). Returns (B,)."""
    summed = counts[:, idx, :].sum(axis=2)          # (n_seeds, B, 3)
    tp, fp, fn = summed[..., 0], summed[..., 1], summed[..., 2]
    return (tp / (tp + fp + fn)).mean(axis=0)

BOOT = {key: {split: boot_seed_mean_iou(COUNTS[key][split], BOOT_IDX[split]) for split in SPLITS}
        for key in METHODS}

rows = []
for key, (label, files) in METHODS.items():
    row = {"method": label, "seeds": len(files)}
    for split in SPLITS:
        ious = seed_ious(COUNTS[key][split])
        lo, hi = np.percentile(BOOT[key][split], [2.5, 97.5])
        f1s = [results[n]["splits"][split]["overall"]["f1"] for n in files]
        row.update({
            f"{split}_iou_mean": ious.mean(),
            f"{split}_iou_ci_low": lo,
            f"{split}_iou_ci_high": hi,
            f"{split}_iou_seed_min": ious.min(),
            f"{split}_iou_seed_max": ious.max(),
            f"{split}_f1_mean": float(np.mean(f1s)),
            f"{split}_water_share": results["fixed_vh"]["splits"][split]["overall"]["water_share"],
        })
    rows.append(row)

final = pd.DataFrame(rows)
final.to_csv(RESULTS_DIR / "final_results.csv", index=False)

# Readable view: IoU mean [95% interval] (seed min to max)
for split in SPLITS:
    print(f"\n{split.upper()} (water share {final[f'{split}_water_share'].iloc[0]:.3f})")
    for _, r in final.iterrows():
        print(f"  {r['method']:<40} {r[f'{split}_iou_mean']:.3f}  [{r[f'{split}_iou_ci_low']:.3f}, "
              f"{r[f'{split}_iou_ci_high']:.3f}]  seeds {r[f'{split}_iou_seed_min']:.3f} to {r[f'{split}_iou_seed_max']:.3f}")
print("\nSaved: final_results.csv")


TEST (water share 0.125)
  Otsu (VH)                                0.230  [0.162, 0.298]  seeds 0.230 to 0.230
  Fixed VH threshold                       0.524  [0.411, 0.612]  seeds 0.524 to 0.524
  U-Net radar, 2 bands                     0.667  [0.545, 0.760]  seeds 0.663 to 0.673
  U-Net optical, 13 bands                  0.820  [0.735, 0.879]  seeds 0.816 to 0.828
  U-Net optical, 6 bands                   0.829  [0.752, 0.882]  seeds 0.827 to 0.832
  Prithvi frozen, 6 bands                  0.824  [0.750, 0.876]  seeds 0.822 to 0.826
  Prithvi full, 6 bands                    0.812  [0.726, 0.871]  seeds 0.812 to 0.812
  U-Net optical, 6 bands, 10% labels       0.828  [0.754, 0.879]  seeds 0.825 to 0.833
  Prithvi frozen, 6 bands, 10% labels      0.791  [0.708, 0.850]  seeds 0.788 to 0.793

BOLIVIA (water share 0.159)
  Otsu (VH)                                0.403  [0.184, 0.612]  seeds 0.403 to 0.403
  Fixed VH threshold                       0.531  [0.372, 0.623]  seeds 0.5

Cell 4  compares pairs of methods on the same chip lists, and shows whether each difference is larger than chip chance.

In [5]:
# Cell 4: Paired differences on the same bootstrap chip lists (A minus B); table saved
PAIRS = [
    ("unet_s2_6", "prithvi_frozen", "Band-matched U-Net vs Prithvi frozen, all labels"),
    ("unet_s2_6_frac10", "prithvi_frozen_frac10", "Band-matched U-Net vs Prithvi frozen, 10% labels"),
    ("prithvi_full", "prithvi_frozen", "Prithvi full vs frozen, all labels (full: 1 seed)"),
    ("unet_s2_6", "unet_s2", "U-Net 6 bands vs 13 bands"),
    ("unet_s2", "unet_s1", "U-Net optical vs radar"),
    ("unet_s1", "fixed", "U-Net radar vs fixed threshold"),
    ("fixed", "otsu", "Fixed threshold vs Otsu"),
    ("unet_s2_6", "unet_s2_6_frac10", "U-Net 6 bands: all labels vs 10%"),
    ("prithvi_frozen", "prithvi_frozen_frac10", "Prithvi frozen: all labels vs 10%"),
]

rows = []
for a, b, title in PAIRS:
    for split in SPLITS:
        diff_point = seed_ious(COUNTS[a][split]).mean() - seed_ious(COUNTS[b][split]).mean()
        lo, hi = np.percentile(BOOT[a][split] - BOOT[b][split], [2.5, 97.5])
        verdict = "A better" if lo > 0 else ("B better" if hi < 0 else "not clear (interval includes 0)")
        rows.append({"comparison": title, "split": split, "A": METHODS[a][0], "B": METHODS[b][0],
                     "diff_A_minus_B": diff_point, "ci_low": lo, "ci_high": hi, "verdict": verdict})

paired = pd.DataFrame(rows)
paired.to_csv(RESULTS_DIR / "paired_differences.csv", index=False)
for _, r in paired.iterrows():
    print(f"{r['comparison']:<52} {r['split']:<8} {r['diff_A_minus_B']:+.3f}  "
          f"[{r['ci_low']:+.3f}, {r['ci_high']:+.3f}]  {r['verdict']}")
print("\nSaved: paired_differences.csv")

Band-matched U-Net vs Prithvi frozen, all labels     test     +0.005  [-0.002, +0.012]  not clear (interval includes 0)
Band-matched U-Net vs Prithvi frozen, all labels     bolivia  +0.036  [-0.014, +0.155]  not clear (interval includes 0)
Band-matched U-Net vs Prithvi frozen, 10% labels     test     +0.037  [+0.018, +0.068]  A better
Band-matched U-Net vs Prithvi frozen, 10% labels     bolivia  +0.012  [-0.036, +0.063]  not clear (interval includes 0)
Prithvi full vs frozen, all labels (full: 1 seed)    test     -0.012  [-0.034, +0.005]  not clear (interval includes 0)
Prithvi full vs frozen, all labels (full: 1 seed)    bolivia  +0.013  [-0.020, +0.069]  not clear (interval includes 0)
U-Net 6 bands vs 13 bands                            test     +0.009  [-0.003, +0.030]  not clear (interval includes 0)
U-Net 6 bands vs 13 bands                            bolivia  +0.016  [+0.002, +0.061]  A better
U-Net optical vs radar                               test     +0.153  [+0.092, +0.226]

Cell 5 saves two tables for the report: the IoU of each test event, and the IoU of the difficult chips, for the main methods.

In [6]:
# Cell 5: Per-event test IoU and difficult-chip IoU (seed means) for the main methods; tables saved
MAIN = ["fixed", "unet_s1", "unet_s2", "unet_s2_6", "prithvi_frozen", "prithvi_full",
        "unet_s2_6_frac10", "prithvi_frozen_frac10"]

# Per-event IoU on test: mean over seeds of the event IoU from summed counts
events = sorted(results["fixed_vh"]["splits"]["test"]["events"])
rows = []
for event in events:
    row = {"event": event, "water_share": results["fixed_vh"]["splits"]["test"]["events"][event]["water_share"]}
    for key in MAIN:
        row[key] = np.mean([results[n]["splits"]["test"]["events"][event]["iou"] for n in METHODS[key][1]])
    rows.append(row)
per_event = pd.DataFrame(rows)
per_event.to_csv(RESULTS_DIR / "per_event_test.csv", index=False)
print("Per-event test IoU (seed means):")
print(per_event.round(3).to_string(index=False))

# Difficult chips: mean chip IoU over seeds; NaN when a chip has no water and no predicted water
DIFFICULT = [("test", "Ghana_866994"), ("test", "Paraguay_34417"),
             ("bolivia", "Bolivia_242570"), ("bolivia", "Bolivia_312675"),
             ("bolivia", "Bolivia_129334"), ("bolivia", "Bolivia_314919"), ("bolivia", "Bolivia_432776")]

def chip_iou(c):
    """IoU of one chip from its counts; NaN when the union is empty."""
    union = c["tp"] + c["fp"] + c["fn"]
    return c["tp"] / union if union > 0 else float("nan")

rows = []
for split, chip in DIFFICULT:
    c0 = results["fixed_vh"]["splits"][split]["chips"][chip]
    scored = c0["tp"] + c0["fp"] + c0["fn"] + c0["tn"]
    row = {"split": split, "chip": chip, "water_share": (c0["tp"] + c0["fn"]) / scored if scored else float("nan")}
    for key in MAIN:
        row[key] = np.nanmean([chip_iou(results[n]["splits"][split]["chips"][chip]) for n in METHODS[key][1]])
    rows.append(row)
difficult = pd.DataFrame(rows)
difficult.to_csv(RESULTS_DIR / "difficult_chips.csv", index=False)
print("\nDifficult chips (seed means):")
print(difficult.round(3).to_string(index=False))
print("\nSaved: per_event_test.csv, difficult_chips.csv")

Per-event test IoU (seed means):
    event  water_share  fixed  unet_s1  unet_s2  unet_s2_6  prithvi_frozen  prithvi_full  unet_s2_6_frac10  prithvi_frozen_frac10
    Ghana        0.080  0.453    0.521    0.493      0.523           0.566         0.482             0.554                  0.560
    India        0.213  0.593    0.704    0.902      0.898           0.889         0.890             0.886                  0.859
   Mekong        0.145  0.724    0.858    0.931      0.930           0.918         0.919             0.926                  0.904
  Nigeria        0.162  0.708    0.885    0.938      0.936           0.911         0.933             0.934                  0.923
 Pakistan        0.180  0.150    0.224    0.679      0.800           0.820         0.700             0.830                  0.644
 Paraguay        0.073  0.397    0.533    0.607      0.600           0.587         0.597             0.599                  0.563
  Somalia        0.065  0.256    0.516    0.807      0.81